## Build a Global Earthquake Tracker App Using Streamlit

<p> The task is to build an interactive Streamlit application that allow users to explore recent global seismic activity using the public USGS Earthquakes API 
This notebook implements Part A of Question 3, data extraction and preprocessing pipeline that builds the Streamlit application (app.py) in Part B. The work is seperated into two files app.py where the application reads a prepared CSV. The cleaned dataset is saved as 'quake_df.csv'. 
 </p>

## Extracting earthquake data

**Task** Fetch the JSON data from the USGS endpoint and extract the following fields from the features array into a pandas DataFrame named 'quake_df' : 'place', 'mag', 'time', 'longitude', 'depth'.

**Method** A single HTTP GET request is sent through the requests libary, the response is parsed as JSON and a loop iterates over each feature to build a list of records. The list is then converted into a pandas DataFrame.


In [1]:
import requests
import pandas as pd

In [2]:
#Check the format beforehand
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"
data = requests.get(url).json()

print("Test feature:")
print(data['features'][0])

print(f"\nTotal features: {len(data['features'])}")

Test feature:
{'type': 'Feature', 'properties': {'mag': 1.08, 'place': '1 km NNW of The Geysers, CA', 'time': 1786023090380, 'updated': 1786023184612, 'tz': None, 'url': 'https://earthquake.usgs.gov/earthquakes/eventpage/nc75412527', 'detail': 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/detail/nc75412527.geojson', 'felt': None, 'cdi': None, 'mmi': None, 'alert': None, 'status': 'automatic', 'tsunami': 0, 'sig': 18, 'net': 'nc', 'code': '75412527', 'ids': ',nc75412527,', 'sources': ',nc,', 'types': ',nearby-cities,origin,phase-data,', 'nst': 24, 'dmin': 0.01304, 'rms': 0.04, 'gap': 61, 'magType': 'md', 'type': 'earthquake', 'title': 'M 1.1 - 1 km NNW of The Geysers, CA'}, 'geometry': {'type': 'Point', 'coordinates': [-122.760665893555, 38.7831649780273, 3.76999998092651]}, 'id': 'nc75412527'}

Total features: 10617


In [3]:
#USGS Earthquakes API endpoint past 30 days of seismic earthquakes.
url = "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_month.geojson"

#fetching JSON from API endpoint
data = requests.get(url).json()

#Building records by looping through features array
earthquakes = []
for feature in data['features']:
    earthquakes.append({
        "place": feature ['properties'] ['place'],
        "mag": feature['properties']['mag'],
        "time": feature['properties']['time'],
        "longitude": feature['geometry']['coordinates'][0],
        "latitude": feature['geometry']['coordinates'][1],
        "depth": feature['geometry']['coordinates'][2]
    })


quake_df = pd.DataFrame(earthquakes)


quake_df.head()

,place,mag,time,longitude,latitude,depth
0,"1 km NNW of The Geysers, CA",1.08,1786023090380,-122.760666,38.783165,3.77
1,"17 km WSW of Johannesburg, CA",0.89,1786022558730,-117.804000,35.312000,6.78
2,"57 km ENE of Chenega, Alaska",1.00,1786019769859,-147.043000,60.244000,14.00
3,"11 km NE of Julian, CA",1.04,1786018542700,-116.524000,33.155500,5.16
4,"66 km WNW of Nanwalek, Alaska",1.40,1786018245393,-153.012000,59.564000,102.20


In [4]:
#quick check to see column types and null counts before any cleaning.
quake_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10617 entries, 0 to 10616
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   place      10617 non-null  object 
 1   mag        10617 non-null  float64
 2   time       10617 non-null  int64  
 3   longitude  10617 non-null  float64
 4   latitude   10617 non-null  float64
 5   depth      10617 non-null  float64
dtypes: float64(4), int64(1), object(1)
memory usage: 497.8+ KB


## Cleaning & transforming the data.
**Task:** 
-Convert the Unix millisecond 'time' field into a proper pandas Datetime.
-Drop rows that have missing values in 'mag' or coordinate columns.
-Create a new column 'risk_category' based on magnitude (Minor <3.0, Moderate 3.0-4.9, Strong >5.0).

**Method** 'pd.to_datetime(..., unit='ms')' converts the timestamp and 'dropna' removes the rows with missing values.


In [5]:
#Convert time from Unix milliseconds to pandas Datetime. 
quake_df['time'] = pd.to_datetime(quake_df['time'], unit='ms')

quake_df.head()

,place,mag,time,longitude,latitude,depth
0,"1 km NNW of The Geysers, CA",1.08,2026-08-06 13:31:30.380,-122.760666,38.783165,3.77
1,"17 km WSW of Johannesburg, CA",0.89,2026-08-06 13:22:38.730,-117.804000,35.312000,6.78
2,"57 km ENE of Chenega, Alaska",1.00,2026-08-06 12:36:09.859,-147.043000,60.244000,14.00
3,"11 km NE of Julian, CA",1.04,2026-08-06 12:15:42.700,-116.524000,33.155500,5.16
4,"66 km WNW of Nanwalek, Alaska",1.40,2026-08-06 12:10:45.393,-153.012000,59.564000,102.20


In [6]:
#Drop any rows that have missing values in the mag (magnitude) or coordinates columns.
quake_df = quake_df.dropna(subset=['mag', 'longitude', 'latitude', 'depth'])

#To verify and check how many rows are left after dropping
print(f"Rows after dropping nulls: {len(quake_df)}")

Rows after dropping nulls: 10617


In [7]:
#Creating a new column called risk_category  based on magnitude with a if loop.
def new_magnitude(mag):
    if mag < 3.0:
        return 'Minor'
    elif mag < 5.0:
        return 'Moderate'
    else:
        return 'Strong'
    
#Apply the function to the magnitude column
quake_df['risk_category'] = quake_df['mag'].apply(new_magnitude)

print(quake_df['risk_category'].value_counts())

risk_category
Minor       9100
Moderate    1296
Strong       221
Name: count, dtype: int64


## Summary

**Task** Print the first 20 rows and the shape of the finale 'quake_df', and save the cleaned data to a CSV file for the Streamlit application.

**Method** .head(20) and .shape gives a confirmation that all columns are correctly typed. Then to save it as a CSV file 'to.csv()'. 

In [8]:
#Print the first 20 rows and shape your final quake_df
print("First 20 rows of quake_df:")
print(quake_df.head(20))

#Print the shape
print(f"\nShape of quake_df: {quake_df.shape}")

First 20 rows of quake_df:
                            place   mag                    time   longitude  \
0     1 km NNW of The Geysers, CA  1.08 2026-08-06 13:31:30.380 -122.760666   
1   17 km WSW of Johannesburg, CA  0.89 2026-08-06 13:22:38.730 -117.804000   
2    57 km ENE of Chenega, Alaska  1.00 2026-08-06 12:36:09.859 -147.043000   
3          11 km NE of Julian, CA  1.04 2026-08-06 12:15:42.700 -116.524000   
4   66 km WNW of Nanwalek, Alaska  1.40 2026-08-06 12:10:45.393 -153.012000   
5           5 km SE of Bishop, CA  2.04 2026-08-06 12:03:56.360 -118.349335   
6      12 km SSW of San Lucas, CA  0.95 2026-08-06 12:01:32.510 -121.051498   
7     22 km E of Balmorhea, Texas  1.20 2026-08-06 11:55:50.194 -103.508000   
8           5 km SE of Bishop, CA  1.86 2026-08-06 11:53:43.600 -118.353996   
9     27 km NNW of Pāhala, Hawaii  1.80 2026-08-06 11:27:56.400 -155.590500   
10     2 km SSW of Loma Linda, CA  1.35 2026-08-06 11:22:32.580 -117.270167   
11  6 km SSE of Warwick, 

In [9]:
quake_df.to_csv("quake_df.csv", index=False)